- **Name:** 20.4_streaming_watermark
- **Author:** Shamas Imran
- **Desciption:** Using watermarks to handle late arriving data in streams
- **Date:** 19-Aug-2025
<!--
REVISION HISTORY
Version          Date        Author           Desciption
01           19-Aug-2025   Shamas Imran       Defined event time column for streaming  
                                              Applied watermark for late data tolerance  
                                              Combined with window operations  
-->

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from pyspark.sql.functions import col, window

StatementMeta(, cce37004-70ee-4eb4-8b50-e3390031fba4, 17, Finished, Available, Finished)

In [16]:
# ------------------------------------------------------------
# 1) Spark Session
# ------------------------------------------------------------
spark = (
    SparkSession.builder
        .appName("Watermark_Drop_vs_Allow_Late_Data")
        .getOrCreate()
)

StatementMeta(, cce37004-70ee-4eb4-8b50-e3390031fba4, 18, Finished, Available, Finished)

In [17]:
# ------------------------------------------------------------
# 2) Folder Paths
# ------------------------------------------------------------
rootPath = "abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Files/"
masterPath = rootPath + "spark-streaming/"
inputPath       = masterPath + "csv_input"
checkpointPath  = masterPath + "checkpoints/watermark"
outputPath      = masterPath + "csv_output"

StatementMeta(, cce37004-70ee-4eb4-8b50-e3390031fba4, 19, Finished, Available, Finished)

In [18]:
# Check if folder exists before deleting
if mssparkutils.fs.exists(masterPath):
    mssparkutils.fs.rm(masterPath, True)  # True = recursive delete

StatementMeta(, cce37004-70ee-4eb4-8b50-e3390031fba4, 20, Finished, Available, Finished)

In [19]:
# Create directories
import os

os.makedirs(masterPath, exist_ok=True)
os.makedirs(inputPath, exist_ok=True)
os.makedirs(checkpointPath, exist_ok=True)
os.makedirs(outputPath, exist_ok=True)

StatementMeta(, cce37004-70ee-4eb4-8b50-e3390031fba4, 21, Finished, Available, Finished)

In [20]:
import pandas as pd
import os


lakehouse_folder = inputPath 

data = [
    {"id": 1, "name": "Ahmed",  "score": 85, "event_time": "2025-08-18 10:00:00"},
    {"id": 2, "name": "Ayesha", "score": 92, "event_time": "2025-08-18 10:00:05"},
    {"id": 3, "name": "Ali",    "score": 78, "event_time": "2025-08-18 10:00:08"},
    {"id": 4, "name": "Fatima", "score": 88, "event_time": "2025-08-18 09:59:55"},
    {"id": 5, "name": "Bilal",  "score": 95, "event_time": "2025-08-18 09:58:45"},
    {"id": 6, "name": "Usman",  "score": 82, "event_time": "2025-08-18 10:02:10"},
    {"id": 7, "name": "Zara",   "score": 90, "event_time": "2025-08-18 10:00:03"},
]


valid_df = pd.DataFrame(data)
valid_path = os.path.join(lakehouse_folder, "Valid_data.csv")
valid_df.to_csv(valid_path, index=False, header=True)

StatementMeta(, cce37004-70ee-4eb4-8b50-e3390031fba4, 22, Finished, Available, Finished)

In [21]:
df = spark.read.format("csv").option("header","true").load(f"{inputPath}/Valid_data.csv")
df.show()

StatementMeta(, cce37004-70ee-4eb4-8b50-e3390031fba4, 23, Finished, Available, Finished)

+---+------+-----+-------------------+
| id|  name|score|         event_time|
+---+------+-----+-------------------+
|  1| Ahmed|   85|2025-08-18 10:00:00|
|  2|Ayesha|   92|2025-08-18 10:00:05|
|  3|   Ali|   78|2025-08-18 10:00:08|
|  4|Fatima|   88|2025-08-18 09:59:55|
|  5| Bilal|   95|2025-08-18 09:58:45|
|  6| Usman|   82|2025-08-18 10:02:10|
|  7|  Zara|   90|2025-08-18 10:00:03|
+---+------+-----+-------------------+



In [22]:
# ------------------------------------------------------------
# 3) Define Schema
# ------------------------------------------------------------
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("event_time", TimestampType(), True)
])

StatementMeta(, cce37004-70ee-4eb4-8b50-e3390031fba4, 24, Finished, Available, Finished)

In [23]:
# ------------------------------------------------------------
# 4) Create Streaming DataFrame
# ------------------------------------------------------------
df_stream = (
    spark.readStream
         .option("header", "true")
         .schema(schema)
         .csv(inputPath)
)

# ------------------------------------------------------------
# 5) Apply Watermark
# ------------------------------------------------------------
# Keep state for 10 minutes; any event older than max(event_time) - 10 minutes is considered late
df_watermarked = df_stream.withWatermark("event_time", "1 minute")

StatementMeta(, cce37004-70ee-4eb4-8b50-e3390031fba4, 25, Finished, Available, Finished)

In [24]:
# ------------------------------------------------------------
# 6) Aggregation Example: Count scores per name
# ------------------------------------------------------------
# Stateful operation needed to demonstrate watermark behavior
agg_df = (
    df_watermarked
        .groupBy("name")
        .count()
        .orderBy("name")
)

StatementMeta(, cce37004-70ee-4eb4-8b50-e3390031fba4, 26, Finished, Available, Finished)

In [25]:
# ------------------------------------------------------------
# 7) Write to Console
# ------------------------------------------------------------
outputPath = "csv_output_watermark"

query = (
    agg_df.writeStream
          .format("delta")                       # ✅ Delta table sink
          .option("checkpointLocation", checkpointPath)
          .outputMode("complete")                # required for aggregations
          .trigger(processingTime="10 seconds")  # micro-batch interval
          .toTable(outputPath)
)

# ------------------------------------------------------------
# 8) Wait for Completion
# ------------------------------------------------------------
query.awaitTermination()


StatementMeta(, cce37004-70ee-4eb4-8b50-e3390031fba4, 27, Finished, Cancelled, Cancelled)

In [26]:
df = spark.sql("SELECT * FROM test_Lakehouse.csv_output_watermark LIMIT 1000")
display(df)

StatementMeta(, cce37004-70ee-4eb4-8b50-e3390031fba4, 28, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, e8bbc183-6d28-4a57-b70d-4809a4be5a88)

In [0]:
"""
id,name,score,event_time
1,Ahmed,85,2025-08-18 10:00:00
2,Ayesha,92,2025-08-18 10:00:05
3,Ali,78,2025-08-18 10:00:08
4,Fatima,88,2025-08-18 09:59:55
5,Bilal,95,2025-08-18 09:58:45
6,Usman,82,2025-08-18 10:02:10
7,Zara,90,2025-08-18 10:00:03

id,name,score,event_time
8,Hassan,89,2025-08-18 10:03:30
9,Imran,91,2025-08-18 09:59:00
10,Laiba,84,2025-08-18 10:02:50
11,Sameer,88,2025-08-18 10:00:30
"""